# Reproducible Kv2.1 sampling-breadth statistics

This notebook recalculates the uncertainty estimates used for the Kv2.1 sampling analyses. It keeps the statistical unit explicit and writes every result to versioned CSV tables.

Three analyses are kept separate:

1. **Nominal first 100 trajectories (L403A):** seed-level normalized global breadth, with a bootstrap over the 20 independent seeds.
2. **Full-QC S6 breadth (WT, L403A, F412L):** trajectory-block bootstrap, keeping all retained recycles from a `(seed, model)` trajectory together.
3. **Full-QC RMSF (WT, L403A, F412L):** trajectory-block bootstrap from the aligned Cα coordinate arrays.

Important provenance note: the current executable, chain-label-invariant S6 estimator does **not** reproduce the historical manuscript point values 3.50, 2.08, and 1.92. Its results below are therefore explicitly labeled **revised current estimates**, not confidence intervals for those historical values.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "shared").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook from within the vgci_mutants repository")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from analysis.statistics_revision.scripts.run_kv21_sampling_breadth_uncertainty import (
    ALL_DISTANCE, BOOTSTRAP_REPLICATES, BOOTSTRAP_SEED, OUT, run
)

pd.set_option("display.max_columns", 50)
results = run()
results

Matplotlib is building the font cache; this may take a moment.


{
  "first100_ratio_summary": "/Users/ahernandezgonzalez/Repositories/vgci_mutants/kv21/dataExtra/conformation_analysis/all_distance_sampling/tables/l403a_first100_seed_level_global_breadth_summary.csv",
  "s6_all_reported_values_reproduced": false,
  "s6_bootstrap_status": "completed for revised current estimator; historical manuscript values remain unreproduced",
  "rmsf_bootstrap_status": "completed from aligned C-alpha arrays with trajectory-block resampling"
}
sequence_background  reported_median_SD_ratio  closest_current_reproducible_median_SD_ratio  representative_only_median_SD_ratio  reported_value_reproduced                                                                      bootstrap_status  bootstrap_median_ratio  bootstrap_95CI_low  bootstrap_95CI_high  fraction_bootstrap_ratio_gt_1  representative_bootstrap_median_ratio  representative_bootstrap_95CI_low  representative_bootstrap_95CI_high  representative_fraction_bootstrap_ratio_gt_1  representative_percent_difference_f

{'first100_ratio_summary': '/Users/ahernandezgonzalez/Repositories/vgci_mutants/kv21/dataExtra/conformation_analysis/all_distance_sampling/tables/l403a_first100_seed_level_global_breadth_summary.csv',
 's6_all_reported_values_reproduced': False,
 's6_bootstrap_status': 'completed for revised current estimator; historical manuscript values remain unreproduced',
 'rmsf_bootstrap_status': 'completed from aligned C-alpha arrays with trajectory-block resampling'}

## 1. Nominal first 100 trajectories: global breadth

The cohort is fixed before QC: the first 20 seeds × 5 model trajectories = 100 nominal trajectories per protocol. Failed trajectories are **not replaced**. For each retained seed, the global breadth is calculated across the complete distance panel; the 20 seed summaries are the independent units.

The reported effect is

\[
G = \frac{\mathrm{median}(B_{\mathrm{masked},s})}
         {\mathrm{median}(B_{\mathrm{vanilla},s})}.
\]

The 95% percentile interval resamples seeds independently within protocol 10,000 times. The Mann–Whitney test and rank-biserial effect size compare the two sets of 20 seed-level breadth values.

In [2]:
first100 = pd.read_csv(ALL_DISTANCE / "l403a_first100_seed_level_global_breadth_summary.csv")
first100_boot = pd.read_csv(ALL_DISTANCE / "l403a_first100_seed_level_global_breadth_ratio_bootstrap.csv")
retention = pd.read_csv(ALL_DISTANCE / "l403a_first100_nominal_trajectory_qc_summary.csv")
display(first100)
display(retention)
print(f"Stored bootstrap replicates: {len(first100_boot):,}")
print("Recomputed percentile CI:", first100_boot["masked_over_vanilla_median_ratio"].quantile([.025, .5, .975]).to_dict())

,vanilla_seeds,masked_seeds,vanilla_median_normalized_seed_IQR,masked_median_normalized_seed_IQR,masked_over_vanilla_median_ratio,median_difference_masked_minus_vanilla,bootstrap_95CI_difference_low,bootstrap_95CI_difference_high,bootstrap_95CI_ratio_low,bootstrap_95CI_ratio_high,bootstrap_ratio_replicates,bootstrap_ratio_seed,mannwhitney_U_masked_vs_vanilla,mannwhitney_p_two_sided,rank_biserial_masked_greater
0,20,20,0.535539,0.805123,1.503388,0.269584,0.193028,0.30591,1.342107,1.603285,10000,20260803,397.0,1.064569e-07,0.985


,protocol,nominal_trajectories,retained_trajectories,first_seed,last_seed,retained_model_recycle_rows,excluded_trajectories,trajectory_retention_fraction
0,vanilla,100,99,55,74,910,1,0.99
1,masked,100,85,33,52,724,15,0.85


Stored bootstrap replicates: 10,000
Recomputed percentile CI: {0.025: 1.3421065360987696, 0.5: 1.4834892895078289, 0.975: 1.6032846753849728}


## 2. Revised current S6 breadth estimator

For each retained structure and each of six model-numbered S6 levels (400, 403, 404, 405, 407, 411), the coordinate is the **maximum of the six inter-subunit Cα ring spans**. Taking a maximum makes this coordinate invariant to arbitrary A/B/C/D chain ordering.

For level `r`, breadth is summarized by

\[
R_r = \frac{SD(D_{r,\mathrm{masked}})}{SD(D_{r,\mathrm{vanilla}})},
\qquad R = \mathrm{median}_{r=1}^{6}(R_r).
\]

`R > 1` means the masked ensemble is broader for the typical S6 level; `R = 1` means equal SD; `R < 1` means vanilla is broader.

The primary bootstrap samples complete `(seed, model)` trajectories with replacement and retains all their QC-passing recycle snapshots as a block. A sensitivity calculation first selects the latest QC representative per trajectory and then resamples those representatives. Both use 10,000 replicates and seed 20260803.

In [3]:
s6 = pd.read_csv(OUT / "kv21_s6_masked_vs_vanilla_breadth_bootstrap.csv")
s6_levels = pd.read_csv(OUT / "kv21_s6_breadth_source_audit.csv")
s6_reps = pd.read_csv(OUT / "kv21_s6_breadth_bootstrap_replicates.csv")

display(s6[[
    "sequence_background", "reported_median_SD_ratio",
    "closest_current_reproducible_median_SD_ratio",
    "bootstrap_95CI_low", "bootstrap_95CI_high",
    "fraction_bootstrap_ratio_gt_1", "representative_only_median_SD_ratio",
    "representative_bootstrap_95CI_low", "representative_bootstrap_95CI_high",
    "representative_percent_difference_from_primary",
    "representative_sensitivity_exceeds_10pct",
    "reported_value_reproduced"
]])
display(s6_levels[[
    "sequence_background", "s6_coordinate_alias", "s6_coordinate_name",
    "vanilla_SD_A", "masked_SD_A", "masked_over_vanilla_SD_ratio",
    "representative_only_SD_ratio"
]])
print(f"Stored S6 bootstrap replicates: {len(s6_reps):,}")

,sequence_background,reported_median_SD_ratio,closest_current_reproducible_median_SD_ratio,bootstrap_95CI_low,bootstrap_95CI_high,fraction_bootstrap_ratio_gt_1,representative_only_median_SD_ratio,representative_bootstrap_95CI_low,representative_bootstrap_95CI_high,representative_percent_difference_from_primary,representative_sensitivity_exceeds_10pct,reported_value_reproduced
0,WT,3.50,2.848771,2.555575,3.081631,1.0,3.067562,2.499601,3.443753,7.680211,False,False
1,L403A,2.08,1.901599,1.709286,2.076714,1.0,1.549884,1.291251,1.856068,-18.495721,True,False
2,F412L,1.92,2.897953,2.647756,3.163493,1.0,2.532573,2.362269,2.761183,-12.608183,True,False


,sequence_background,s6_coordinate_alias,s6_coordinate_name,vanilla_SD_A,masked_SD_A,masked_over_vanilla_SD_ratio,representative_only_SD_ratio
0,WT,V398,S6_cross_pore_max_400,0.188184,0.517734,2.751206,2.142579
1,WT,I401,S6_cross_pore_max_403,0.250079,2.244222,8.974064,14.239182
2,WT,A402,S6_cross_pore_max_404,0.168339,0.497687,2.956459,4.055911
3,WT,L403/A403,S6_cross_pore_max_405,0.195574,0.330620,1.690510,1.971422
4,WT,I405,S6_cross_pore_max_407,0.372994,0.520185,1.394622,1.223445
5,WT,V409,S6_cross_pore_max_411,0.393010,1.157938,2.946335,3.992546
6,L403A,V398,S6_cross_pore_max_400,0.187278,0.366042,1.954539,1.362640
7,L403A,I401,S6_cross_pore_max_403,0.303000,1.994749,6.583327,7.337139
8,L403A,A402,S6_cross_pore_max_404,0.208722,0.385856,1.848659,1.937555
9,L403A,L403/A403,S6_cross_pore_max_405,0.210115,0.279587,1.330638,1.211500


Stored S6 bootstrap replicates: 60,000


### S6 interpretation guardrail

The percentile intervals quantify uncertainty for the **revised current estimator shown here**. They must not be attached to the historical 3.50/2.08/1.92 values because those point estimates are not reproduced by the current source tables and executable selector. The representative-only result is a sensitivity analysis, not a replacement estimand.

## 3. RMSF masked-minus-vanilla differences

RMSF is recomputed from aligned Cα coordinate arrays after every bootstrap draw. Complete `(seed, model)` trajectories are sampled as blocks, preserving all retained recycles. The effect is the median residue-wise difference

\[
\Delta = \mathrm{median}(RMSF_{masked} - RMSF_{vanilla})
\]

reported separately for directly masked positions and positions outside the direct mask. Positive values mean greater RMSF under masking.

In [4]:
rmsf = pd.read_csv(OUT / "kv21_rmsf_trajectory_block_bootstrap.csv")
rmsf_audit = pd.read_csv(OUT / "kv21_rmsf_bootstrap_source_audit.csv")
display(rmsf)
display(rmsf_audit)

,sequence_background,region,observed_median_masked_minus_vanilla_RMSF_A,bootstrap_median_A,bootstrap_95CI_low_A,bootstrap_95CI_high_A,bootstrap_replicates,bootstrap_seed,bootstrap_unit
0,WT,directly_masked_positions,1.467352,1.466091,1.203577,1.732275,10000,20260803,complete model-seed trajectory
1,WT,outside_direct_mask,-0.240177,-0.186130,-0.728088,0.175543,10000,20260803,complete model-seed trajectory
2,L403A,directly_masked_positions,1.307682,1.316377,1.036825,1.582632,10000,20260803,complete model-seed trajectory
3,L403A,outside_direct_mask,-0.395433,-0.377352,-1.040971,0.116338,10000,20260803,complete model-seed trajectory
4,F412L,directly_masked_positions,1.306238,1.311206,1.117422,1.507174,10000,20260803,complete model-seed trajectory
5,F412L,outside_direct_mask,-0.472332,-0.464234,-0.840170,-0.011465,10000,20260803,complete model-seed trajectory


,sequence_background,vanilla_trajectories,masked_trajectories,coordinate_source,selection_source
0,WT,490,458,kv21/dataRMSF/merged/kv21_aligned_ca_coordinat...,six *_all_ok_rmsd_3A_structural_interface_qc.c...
1,L403A,484,459,kv21/dataRMSF/merged/kv21_aligned_ca_coordinat...,six *_all_ok_rmsd_3A_structural_interface_qc.c...
2,F412L,487,448,kv21/dataRMSF/merged/kv21_aligned_ca_coordinat...,six *_all_ok_rmsd_3A_structural_interface_qc.c...


## 4. Reproducibility checks and output inventory

In [5]:
assert BOOTSTRAP_REPLICATES == 10_000
assert BOOTSTRAP_SEED == 20260803
assert len(first100_boot) == 10_000
assert len(s6_reps) == 3 * 2 * 10_000
assert not s6["reported_value_reproduced"].any()
assert (s6["fraction_bootstrap_ratio_gt_1"] == 1).all()
assert (s6["qualitative_conclusion_agrees_ratio_gt_1"]).all()

outputs = sorted(path.relative_to(ROOT).as_posix() for path in OUT.glob("kv21_*"))
print("All checks passed.\n")
print("\n".join(outputs))

All checks passed.

analysis/statistics_revision/tables/kv21_rmsf_bootstrap_source_audit.csv
analysis/statistics_revision/tables/kv21_rmsf_trajectory_block_bootstrap.csv
analysis/statistics_revision/tables/kv21_s6_breadth_bootstrap_replicates.csv
analysis/statistics_revision/tables/kv21_s6_breadth_source_audit.csv
analysis/statistics_revision/tables/kv21_s6_masked_vs_vanilla_breadth_bootstrap.csv
analysis/statistics_revision/tables/kv21_sampling_breadth_manuscript_statistics.csv
analysis/statistics_revision/tables/kv21_sampling_breadth_uncertainty_run_summary.json
